# Encoder arm: fine-tuned BERT-family models

Runtime → Change runtime type → **T4 GPU** (or L4). Then Runtime → Run all.

Results are written straight to `MyDrive/capstone_runs/results/`, so finished methods survive a disconnect. Re-running skips nothing: each method's rows are replaced, other rows kept.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO = 'https://github.com/ROLFFFX/operating-point-triage.git'  # set after the repo is created
%cd /content
!rm -rf operating-point-triage && git clone -q $REPO
%cd /content/operating-point-triage
!pip install -q -r requirements.txt -r requirements-gpu.txt

In [ ]:
# Keep results on Drive: seed it with the repo's baseline CSVs once, then link.
RUNS = '/content/drive/MyDrive/capstone_runs'
!mkdir -p $RUNS/results
!rsync -a --ignore-existing results/ $RUNS/results/
!rm -rf results && ln -s $RUNS/results results
!ls results

In [ ]:
!python -m scripts.download_data

## 1. Quick check (~1 min): tiny model, one epoch
If this fails, stop and send me the error.

In [ ]:
!python -m scripts.run_encoders --models tiny --heads binary --epochs 1 --max-len 128 --n-boot 50 --tag smoke

## 2. Main run, seed 0 (~1 h on a T4)
Bio_ClinicalBERT and RoBERTa-base, binary and ordinal heads, 5-fold CV, appended to `results/primary`.

In [ ]:
!python -m scripts.run_encoders --models bioclinicalbert roberta --heads binary ordinal --model-seed 0 --tag primary

## 3. Extra seeds for training variance (~2 h); run if units allow

In [ ]:
for s in (1, 2):
    !python -m scripts.run_encoders --models bioclinicalbert roberta --heads binary ordinal --model-seed {s} --tag seeds/s{s}

## 4. Sensitivity definition (Levels 1–2), seed 0 (~1 h); optional

In [ ]:
!python -m scripts.run_encoders --models bioclinicalbert roberta --heads binary ordinal --max-level 2 --tag sens_level2

## 5. Bundle for download
Download `encoder_results.zip` from the Files pane (or it is already in `MyDrive/capstone_runs/`).

In [ ]:
!cd $RUNS && zip -qr encoder_results.zip results -x '*/smoke/*' && ls -lh $RUNS/encoder_results.zip